# Day 4: グリッドネットワークとマクロ基本図 (MFD)

## 学習目標
- グリッドネットワークで渋滞がどう伝播するかを可視化する
- マクロ基本図（Macroscopic Fundamental Diagram, MFD）の考え方と、
  `compute_mfd` / `macroscopic_fundamental_diagram` / `mfd_to_pandas` の使い方を身につける
- 需要レベルを変えたときに、ネットワーク全体の「平均密度 vs 平均流率」の関係がどう変わるかを観察する

## 前提知識
- [`docs/theory.md`](../docs/theory.md) 1節「基本図」の復習（1リンクの q-k 関係）
- MFDは、この基本図を「ネットワーク全体」に拡張した概念です:
  個々のリンクではなく、エリア全体の平均密度・平均流率をプロットすると、
  ある密度までは流率が上がり、過密になると急激に下がる、という
  ネットワークスケールの「基本図」が現れることが知られています。


In [ ]:
from uxsim import World
import random

def build_grid_world(demand_flow=0.25, tmax=3600, imax=4, jmax=4, seed=0):
    W = World(
        name="",
        deltan=5,
        tmax=tmax,
        print_mode=0, save_mode=1, show_mode=0,
        random_seed=seed,
    )
    nodes = {}
    for i in range(imax):
        for j in range(jmax):
            nodes[i, j] = W.addNode(f"n{(i,j)}", i, j, flow_capacity=1.6)

    for i in range(imax):
        for j in range(jmax):
            if i != imax - 1:
                W.addLink(f"l{(i,j,i+1,j)}", nodes[i, j], nodes[i+1, j], length=1000, free_flow_speed=20, number_of_lanes=1)
            if i != 0:
                W.addLink(f"l{(i,j,i-1,j)}", nodes[i, j], nodes[i-1, j], length=1000, free_flow_speed=20, number_of_lanes=1)
            if j != jmax - 1:
                W.addLink(f"l{(i,j,i,j+1)}", nodes[i, j], nodes[i, j+1], length=1000, free_flow_speed=20, number_of_lanes=1)
            if j != 0:
                W.addLink(f"l{(i,j,i,j-1)}", nodes[i, j], nodes[i, j-1], length=1000, free_flow_speed=20, number_of_lanes=1)

    rng = random.Random(seed)
    node_list = list(nodes.values())
    for _ in range(imax * jmax):  # 全ノードペアの一部からランダムにOD需要を張る
        o, d = rng.sample(node_list, 2)
        W.adddemand(o, d, t_start=0, t_end=tmax * 0.7, flow=demand_flow / (imax * jmax))
    return W

W = build_grid_world(demand_flow=6.0)
W.exec_simulation()
W.analyzer.print_simple_stats()


In [ ]:
# ネットワークスナップショット: 渋滞がどこから始まり、どう伝播するかを目視で確認
for t in [600, 1800, 3000]:
    W.analyzer.network(t=t, detailed=1, network_font_size=8)


In [ ]:
# マクロ基本図(MFD)をプロット: 横軸=ネットワーク平均密度、縦軸=ネットワーク平均流率
W.analyzer.macroscopic_fundamental_diagram(kappa=0.15, qmax=0.6)

df_mfd = W.analyzer.mfd_to_pandas()
df_mfd.head()


## Part B: 演習

1. `demand_flow` を `3.0`（余裕あり）・`6.0`（今回の例）・`12.0`（過密）の3水準で実行し、
   MFDの形がどう変わるか比較してください。過密にした場合、MFDの右側（高密度側）で
   流率がどうなるか観察しましょう（グリッドロック現象）。
2. 需要の立ち上がり方（`t_start`/`t_end`）を変えて、「負荷が増えていく過程」と
   「負荷が減っていく過程」でMFDの軌跡が同じ経路を通るか、それとも
   ヒステリシス（往復で異なる経路をたどる現象）が見られるかを確認してください。


In [ ]:
# TODO: 需要レベルを変えて比較する
# W_low = build_grid_world(demand_flow=3.0)
# W_high = build_grid_world(demand_flow=12.0)
# それぞれ exec_simulation() し、macroscopic_fundamental_diagram() で見比べる


## Part C: 考察

- MFDが「ある密度を超えると急激に流率が落ちる」形になるのはなぜでしょうか。
  1リンクの三角形基本図（Day 1）と、ネットワーク全体のMFDの関係を自分の言葉で説明してください。
- 実際の都市の交通管制（例: 流入制御・エリア規制）は、MFDのどの部分を避けるための施策と
  言えるか考えてみてください。
